In [0]:
fire_csv_path = "/Volumes/samples/databricks/datasets/learning-spark-v2/sf-fire/sf-fire-calls.csv"
retail_data_csv_path = "/Volumes/samples/databricks/datasets/online_retail/data-001/data.csv"
asa_airlines_csv_path = "/Volumes/samples/databricks/datasets/asa/airlines/2005.csv"

In [0]:
fire_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(fire_csv_path)
retail_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(retail_data_csv_path)

In [0]:
import re

def fix_column_names(df):
    for col in df.columns:
        new_col = re.sub(r"[ ,;{}()\n\t-]","_",col)
        df = df.withColumnRenamed(col, new_col)
    return df

In [0]:
fire_df_fixed = fix_column_names(fire_df)

In [0]:
fire_df_fixed.columns

Caching Understanding in case of Delta lake : Cluster is Delta Cache Accelerated

In [0]:
spark.conf.get("spark.databricks.io.cache.enabled")

In [0]:
airlines_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(asa_airlines_csv_path)

In [0]:
display(airlines_df)

In [0]:
spark.conf.set("spark.databricks.io.cache.enabled",False)

In [0]:
spark.conf.get("spark.databricks.io.cache.enabled")

In [0]:
airlines_df.write.format("delta").mode("overwrite").option("path","abfss://data@dbcertificationstg.dfs.core.windows.net/deltalaketables").saveAsTable("dbcertification.deltalakeschema.airlines_2025_stream_test")

In [0]:
%sql
select * from dbcertification.deltalakeschema.airlines_2025_stream_test where UniqueCarrier="UA";

In [0]:
spark.conf.set("spark.databricks.io.cache.enabled",True)

In [0]:
spark.conf.get("spark.databricks.io.cache.enabled")

In [0]:
%sql
select * from dbcertification.deltalakeschema.airlines_2025_stream_test where UniqueCarrier="UA";

FILE OPTIMIZE - OPTIMIZATION

In [0]:
files = dbutils.fs.ls("abfss://data@dbcertificationstg.dfs.core.windows.net/deltalaketables/")

In [0]:
from pyspark.sql.functions import col, round

files_df = spark.createDataFrame(files)
files_df = files_df.withColumn("file_size_mb",round(col("size") / (1024 * 1024),2))
display(files_df)

In [0]:
from delta.tables import DeltaTable
delta_table_airlines = DeltaTable.forPath(spark,"abfss://data@dbcertificationstg.dfs.core.windows.net/deltalaketables/")

In [0]:
delta_table_airlines.optimize().executeCompaction()

116 mb of 1 file is created

In [0]:
files = dbutils.fs.ls("abfss://data@dbcertificationstg.dfs.core.windows.net/deltalaketables/")
files_df = spark.createDataFrame(files)
files_df = files_df.withColumn("file_size_mb",round(col("size") / (1024 * 1024),2))
display(files_df)

#### Z-ORDER